# Week 3 — RAG Evaluation — All 4 Tasks (Base + LoRA+RAG variants)

**Self-contained. Evaluates both Base RAG and LoRA+RAG for direct 4-way comparison.**

| | HumanEval+HumanEvalX | MBPP |
|---|---|---|
| NL→Python | ✅ Real Python test | ✅ Real Python asserts |
| NL→Java | ✅ Real Java test (HEX) | ⚠️ Extended CSV java_test |
| Python→Java | ✅ Real Java test (HEX) | ⚠️ Extended CSV java_test |
| Code→NL | ✅ Human NL | ✅ Human NL |

**Same 33 problems as LoRA notebook (SEED=13) across all 4 tasks per dataset.**

**Run twice:**
- `USE_LORA = False` → Base model RAG
- `USE_LORA = True` → LoRA fine-tuned model + RAG

**Settings:** Qwen 1.5B · K=[0,1] · 3 retries · adapter from `lora_finetuned/`

**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Setup

In [ ]:
# Drive mount
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CANDIDATES = [
        '/content/drive/MyDrive/codegen_week1',
        '/content/drive/MyDrive/Shared with me/codegen_week1',
    ]
    PROJECT_DIR = next((Path(p) for p in CANDIDATES if Path(p).exists()), Path(CANDIDATES[0]))
except Exception:
    PROJECT_DIR = Path('/tmp/codegen_week1')

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PROJECT_DIR / 'results'
RAG_DIR     = PROJECT_DIR / 'rag_final'
RESULTS_DIR.mkdir(exist_ok=True)
RAG_DIR.mkdir(exist_ok=True)
print(f'PROJECT_DIR : {PROJECT_DIR}')
print(f'RAG_DIR     : {RAG_DIR}')
print(f'RESULTS_DIR : {RESULTS_DIR}')

Mounted at /content/drive
PROJECT_DIR : /content/drive/MyDrive/codegen_week1
RAG_DIR     : /content/drive/MyDrive/codegen_week1/rag_final
RESULTS_DIR : /content/drive/MyDrive/codegen_week1/results


In [ ]:
# Install deps
import subprocess, sys

def sh(cmd): subprocess.run(cmd, shell=True, check=False)

sh('apt-get -qq install -y openjdk-17-jdk-headless > /dev/null 2>&1')
sh('java -version')
sh(f'{sys.executable} -m pip -q install -U '
   '"transformers>=4.44" "accelerate>=0.33" "bitsandbytes>=0.43" '
   '"peft>=0.10" '
   '"datasets>=2.20" "huggingface_hub>=0.23" '
   '"faiss-cpu" "sentence-transformers" '
   '"rouge-score" "javalang" "pandas" "tqdm"')
print('Done.')

Done.


In [ ]:
# Config
import os
from pathlib import Path

SEED        = int(os.environ.get('CODEGEN_SEED',  '13'))
EVAL_SEED   = SEED              # same seed as LoRA notebook for fair comparison
MODEL_SIZE  = os.environ.get('CODEGEN_MODEL', '1.5b')
SAMPLE      = int(os.environ.get('CODEGEN_SAMPLE', '33'))  # match LoRA N_TEST=33
K_VALUES    = [0, 1]            # 0=zero-shot, 1=RAG
N_TRIES     = 3                 # retry attempts (same as LoRA notebook)
EXEC_TIMEOUT_S  = 15
EMBED_MODEL_ID  = 'sentence-transformers/all-MiniLM-L6-v2'

# LoRA config
# Set USE_LORA=True  → loads LoRA adapter → produces LoRA+RAG results
# Set USE_LORA=False → base model only   → produces Base RAG results
USE_LORA    = True   # Change this between runs
ADAPTER_DIR = PROJECT_DIR / 'lora_finetuned'

MODELS = {
    '1.5b': 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    '7b':   'Qwen/Qwen2.5-Coder-7B-Instruct',
}
MODEL_ID = MODELS[MODEL_SIZE]
TASKS    = ['nl2py', 'nl2java', 'py2java', 'code2nl']

# Output tag — used in all saved filenames
RUN_TAG  = 'lora_rag' if USE_LORA else 'base_rag'

print(f'Model       : {MODEL_ID}')
print(f'USE_LORA    : {USE_LORA}  →  RUN_TAG={RUN_TAG}')
print(f'ADAPTER_DIR : {ADAPTER_DIR}')
print(f'  Exists    : {ADAPTER_DIR.exists()}')
print(f'SAMPLE      : {SAMPLE} problems per dataset (same 33 as LoRA notebook)')
print(f'K_VALUES    : {K_VALUES}')
print(f'EMBED_MODEL : {EMBED_MODEL_ID}')

if USE_LORA and not ADAPTER_DIR.exists():
    print()
    print('   WARNING: ADAPTER_DIR not found.')
    print('   Copy lora_finetuned/ to your Drive codegen_week1/ folder first.')

Model       : Qwen/Qwen2.5-Coder-1.5B-Instruct
USE_LORA    : True  →  RUN_TAG=lora_rag
ADAPTER_DIR : /content/drive/MyDrive/codegen_week1/lora_finetuned
  Exists    : True
SAMPLE      : 33 problems per dataset (same 33 as LoRA notebook)
K_VALUES    : [0, 1]
EMBED_MODEL : sentence-transformers/all-MiniLM-L6-v2


In [ ]:
# GPU check
import torch, warnings
warnings.filterwarnings('ignore', category=FutureWarning)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM: {free/1e9:.1f} / {total/1e9:.1f} GB')
else:
    raise SystemError('No GPU — switch to T4 runtime')

CUDA: True
GPU: Tesla T4
VRAM: 15.5 / 15.6 GB


## 2. Execution Sandboxes *(identical to LoRA notebook)*

In [ ]:
# Python sandbox
import subprocess, sys, tempfile
from pathlib import Path

def run_python(full_program: str, timeout: int = EXEC_TIMEOUT_S) -> dict:
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / 'solution.py'
        path.write_text(full_program, encoding='utf-8')
        try:
            res = subprocess.run(
                [sys.executable, str(path)],
                capture_output=True, text=True, timeout=timeout,
                env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'},
            )
            if res.returncode == 0:
                return {'passed': True, 'error': None}
            return {'passed': False, 'error': (res.stderr or res.stdout)[:500]}
        except subprocess.TimeoutExpired:
            return {'passed': False, 'error': f'timeout >{timeout}s'}
        except Exception as e:
            return {'passed': False, 'error': repr(e)}

In [ ]:
# Java sandbox
import re as _re

_CLASS_RE    = _re.compile(r'public\s+class\s+(\w+)')
_JAVA_IMPORTS = (
    'import java.util.*;\n'
    'import java.util.stream.*;\n'
    'import java.util.regex.*;\n'
    'import java.lang.*;\n'
    'import java.math.*;\n'
)

def _inject_imports(src: str) -> str:
    head = src.lstrip()
    if head.startswith('package '):
        nl = head.find('\n')
        return head[:nl+1] + _JAVA_IMPORTS + head[nl+1:]
    return _JAVA_IMPORTS + head

def validate_java(solution_src: str, test_src: str, timeout: int = EXEC_TIMEOUT_S) -> dict:
    if not solution_src or not solution_src.strip():
        return {'passed': False, 'error': 'empty solution'}
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        sol  = _inject_imports(solution_src)
        m    = _CLASS_RE.search(sol)
        sc   = m.group(1) if m else 'Solution'
        (tmp / f'{sc}.java').write_text(sol, encoding='utf-8')
        test = _inject_imports(test_src)
        m2   = _CLASS_RE.search(test)
        tc   = m2.group(1) if m2 else 'Main'
        (tmp / f'{tc}.java').write_text(test, encoding='utf-8')
        cres = subprocess.run(['javac', f'{sc}.java', f'{tc}.java'],
                              capture_output=True, text=True, cwd=tmp, timeout=30)
        if cres.returncode != 0:
            return {'passed': False, 'error': 'COMPILE: ' + cres.stderr[:400]}
        rres = subprocess.run(['java', '-ea', '-cp', '.', tc],
                              capture_output=True, text=True, cwd=tmp, timeout=timeout)
        if rres.returncode == 0:
            return {'passed': True, 'error': None}
        return {'passed': False, 'error': 'RUNTIME: ' + (rres.stderr or rres.stdout)[:400]}

_smoke = validate_java(
    'public class Solution { public static int add(int a,int b){return a+b;} }',
    'public class Main { public static void main(String[] a){ assert Solution.add(2,3)==5; } }'
)
print('Java sandbox:', 'PASS' if _smoke['passed'] else f'FAIL: {_smoke["error"]}')

Java sandbox: PASS


## 3. Block Extractors, Validators & Generation

In [ ]:
# block extractors
import re

def _strip_fences(text: str, lang_hints: tuple) -> str:
    s = text.strip()
    open_re = re.compile(r'```(?:' + '|'.join(lang_hints) + r')?\s*\n?', re.IGNORECASE)
    m = open_re.search(s)
    if not m:
        return (s + '\n') if s else ''
    body = s[m.end():]
    close = re.search(r'```', body)
    body = body[:close.start()] if close else body
    return (body.rstrip() + '\n') if body.rstrip() else ''

def extract_python_body(text): return _strip_fences(text, ('python', 'py', ''))
def extract_java_body(text):   return _strip_fences(text, ('java', ''))

def extract_solution_and_test(text: str):
    blocks = re.findall(r'```java\s*\n(.*?)```', text, re.S | re.I)
    if len(blocks) >= 2: return blocks[0].strip(), blocks[1].strip()
    if len(blocks) == 1: return blocks[0].strip(), ''
    return '', ''

def strip_markdown(text: str) -> str:
    t = re.sub(r'```[a-zA-Z]*\n.*?```', '', text, flags=re.S)
    return t.strip()

In [ ]:
# AST helpers
import ast as _ast

def py_docstring(src: str) -> str:
    try: tree = _ast.parse(src)
    except SyntaxError:
        try: tree = _ast.parse(src.rstrip() + '\n    pass\n')
        except: return ''
    for node in _ast.walk(tree):
        if isinstance(node, (_ast.FunctionDef, _ast.AsyncFunctionDef)):
            doc = _ast.get_docstring(node)
            if doc: return doc.strip()
    return ''

def strip_py_docstring(src: str) -> str:
    try: tree = _ast.parse(src)
    except: return src
    for node in _ast.walk(tree):
        if isinstance(node, (_ast.FunctionDef, _ast.AsyncFunctionDef,
                              _ast.ClassDef, _ast.Module)):
            body = getattr(node, 'body', [])
            if (body and isinstance(body[0], _ast.Expr)
                    and isinstance(body[0].value, _ast.Constant)
                    and isinstance(body[0].value.value, str)):
                node.body = body[1:] or [_ast.Pass()]
    try: return _ast.unparse(tree)
    except: return src

def py_signature(py_src: str) -> str:
    for line in py_src.splitlines():
        if re.match(r'\s*def\s+\w+', line): return line.strip()
    return 'def solution(*args):'

In [ ]:
# generation + retry loop (identical to LoRA notebook)
import torch

def llm(tokenizer, model, user_prompt: str, max_new_tokens: int = 512) -> str:
    msgs = [{'role': 'user', 'content': user_prompt}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False,
            max_new_tokens=max_new_tokens,
        )
    new_ids = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

def run_with_retries(generate_fn, validate_fn, feedback_fn, n_tries=N_TRIES):
    best     = {'flag': False, 'struct': {}, 'verdict': {'passed': False}, 'raw': ''}
    feedback = None
    for attempt in range(1, n_tries + 1):
        raw, struct = generate_fn(feedback)
        verdict     = validate_fn(struct)
        if verdict['passed']:
            return {'flag': True,
                    'best': {'struct': struct, 'verdict': verdict, 'raw': raw},
                    'n_attempts': attempt}
        if not best['flag']:
            best = {'flag': False, 'struct': struct, 'verdict': verdict, 'raw': raw}
        feedback = feedback_fn(struct, verdict.get('error', '') or '')
    return {'flag': False, 'best': best, 'n_attempts': n_tries}

## 4. Load Evaluation Problems

**HumanEval+HumanEvalX:** Fresh from HuggingFace — real NL, real Python tests, real Java tests.

**MBPP:** NL + Python + py_test from HuggingFace. Java + java_test joined from extended CSV (flag=True).

**Same 33 problems used across all 4 tasks per dataset (matches LoRA notebook N_TEST=33, SEED=13).**

In [ ]:
# HumanEval-X split loader
import gzip, json as _json, random
from huggingface_hub import hf_hub_download

def _hex_split(lang: str) -> list:
    REPO = 'THUDM/humaneval-x'
    candidates = [
        f'data/{lang}/data/humaneval.jsonl',
        f'data/{lang}/data/humaneval_{lang}.jsonl.gz',
        f'data/{lang}/humaneval_{lang}.jsonl.gz',
        f'{lang}/data/humaneval_{lang}.jsonl.gz',
        f'data/{lang}/data/humaneval.jsonl.gz',
    ]
    last_err = None
    for filename in candidates:
        try:
            path = hf_hub_download(repo_id=REPO, filename=filename, repo_type='dataset')
            opener = gzip.open if filename.endswith('.gz') else open
            with opener(path, 'rt', encoding='utf-8') as f:
                return [_json.loads(line) for line in f if line.strip()]
        except Exception as e:
            last_err = e
    raise RuntimeError(f'Could not load HumanEval-X/{lang}: {last_err!r}')

def _sample(rows, n, seed=EVAL_SEED):
    rows = list(rows)
    random.Random(seed).shuffle(rows)
    return rows[:n]

In [ ]:
# Load HumanEval + HumanEval-X merged (same SEED=13 as LoRA notebook)
from datasets import load_dataset as _load_ds

print('Loading HumanEval-X from HuggingFace...')
py_hex = _hex_split('python')
ja_hex = _hex_split('java')
by_id_py = {r['task_id'].split('/')[-1]: r for r in py_hex}
by_id_ja = {r['task_id'].split('/')[-1]: r for r in ja_hex}
print(f'HEX python: {len(by_id_py)} | java: {len(by_id_ja)}')

print('Loading HumanEval from HuggingFace...')
try:
    he_ds = _load_ds('openai/openai_humaneval', split='test')
except Exception:
    he_ds = _load_ds('openai_humaneval', split='test', trust_remote_code=True)

he_all = []
for r in he_ds:
    k  = r['task_id'].split('/')[-1]
    rp = by_id_py.get(k, {})
    rj = by_id_ja.get(k, {})
    if not rj: continue
    he_all.append({
        'id':             r['task_id'],
        'num':            k,
        'source':         'humaneval',
        'nl':             py_docstring(r['prompt']) or r['prompt'].strip(),
        'py_canonical':   r['prompt'] + r['canonical_solution'],
        'py_test':        r['test'],
        'py_entry':       r['entry_point'],
        'java_canonical': rj.get('prompt','') + rj.get('canonical_solution',''),
        'java_decl':      rj.get('declaration', rj.get('prompt','')),
        'java_test':      rj.get('test',''),
    })

# Use EVAL_SEED=13 — same as LoRA notebook SEED
he_eval = _sample(he_all, SAMPLE, seed=EVAL_SEED)
print(f'HumanEval merged: {len(he_all)} total, {len(he_eval)} sampled')
print(f'Sample: {he_eval[0]["id"]} | py_entry={he_eval[0]["py_entry"]} | nl={he_eval[0]["nl"][:50]}')

Loading HumanEval-X from HuggingFace...


humaneval.jsonl:   0%|          | 0.00/343k [00:00<?, ?B/s]

humaneval.jsonl:   0%|          | 0.00/475k [00:00<?, ?B/s]

HEX python: 164 | java: 164
Loading HumanEval from HuggingFace...


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

HumanEval merged: 164 total, 33 sampled
Sample: HumanEval/79 | py_entry=decimal_to_binary | nl=You will be given a number in decimal form and you


In [ ]:
# Load MBPP eval problems
# NL + Python + py_test: from HuggingFace (real ground truth)
# Java + java_test: from extended MBPP CSV (flag=True validated in week 2)
import pandas as pd

print('Loading MBPP from HuggingFace...')
try:
    mbpp_ds = _load_ds('google-research-datasets/mbpp', 'sanitized', split='test')
except Exception:
    mbpp_ds = _load_ds('mbpp', 'sanitized', split='test', trust_remote_code=True)

mbpp_ext = {}
mbpp_csv = RESULTS_DIR / f'extended_mbpp_seed{SEED}.csv'
if mbpp_csv.exists():
    df_m = pd.read_csv(mbpp_csv)
    df_m = df_m[df_m['flag'] == True]
    for _, r in df_m.iterrows():
        pid = str(int(r['problem_id']))
        mbpp_ext[pid] = {
            'java':      str(r.get('java', '') or ''),
            'java_test': str(r.get('java_test', '') or ''),
        }
    print(f'Extended MBPP CSV: {len(mbpp_ext)} flag=True rows')
else:
    print(f'WARNING: {mbpp_csv} not found — NL->Java/Python->Java will use on-the-fly generation')

mbpp_all = []
for r in mbpp_ds:
    pid  = str(r['task_id'])
    ext  = mbpp_ext.get(pid, {})
    mbpp_all.append({
        'id':             pid,
        'num':            pid,
        'source':         'mbpp',
        'nl':             r['prompt'],
        'py_canonical':   r['code'],
        'py_test':        '\n'.join(r['test_list']),
        'py_entry':       '',
        'java_canonical': ext.get('java', ''),
        'java_decl':      '',
        'java_test':      ext.get('java_test', ''),
    })

mbpp_with_java = [r for r in mbpp_all if r['java_canonical'].strip()]
mbpp_eval = _sample(mbpp_with_java, SAMPLE, seed=EVAL_SEED)
print(f'MBPP: {len(mbpp_all)} total | {len(mbpp_with_java)} with Java | {len(mbpp_eval)} sampled')
print(f'Sample: id={mbpp_eval[0]["id"]} | nl={mbpp_eval[0]["nl"][:50]}')

Loading MBPP from HuggingFace...


README.md:   0%|          | 0.00/9.06k [00:00<?, ?B/s]

sanitized/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 33.9kB            

sanitized/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sanitized/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 60.9kB            

...

Generating train split:   0%|          | 0/120 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/257 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/43 [00:00<?, ? examples/s]

Generating prompt split:   0%|          | 0/7 [00:00<?, ? examples/s]

Extended MBPP CSV: 216 flag=True rows
MBPP: 257 total | 216 with Java | 33 sampled
Sample: id=223 | nl=Write a function that takes in a sorted array, its


## 5. FAISS Corpus & Indexes

Corpus = extended CSVs (all 3 datasets, flag=True only).

- `nl_index` — embeds NL → for NL→Python, NL→Java retrieval
- `py_index` — embeds Python → for Python→Java, Code→NL retrieval

**Index is shared between Base RAG and LoRA+RAG runs** — FAISS is built once and reused.

In [ ]:
# build corpus from extended CSVs (flag=True only)
from collections import Counter

corpus = []

def _add_corpus(csv_name, source_tag):
    path = RESULTS_DIR / csv_name
    if not path.exists():
        print(f'WARNING: {csv_name} not found — skipping')
        return
    df = pd.read_csv(path)
    df = df[df['flag'] == True]
    added = 0
    for _, r in df.iterrows():
        py = str(r.get('python','') or '').strip()
        if not py: continue
        corpus.append({
            'id':             str(r.get('problem_id','')),
            'source':         source_tag,
            'nl':             str(r.get('nl','') or '').strip(),
            'py_canonical':   py,
            'java_canonical': str(r.get('java','') or '').strip(),
        })
        added += 1
    print(f'{csv_name}: {added} rows added')

_add_corpus(f'extended_humaneval_seed{SEED}.csv',   'humaneval')
_add_corpus(f'extended_humaneval_x_seed{SEED}.csv', 'humaneval_x')
_add_corpus(f'extended_mbpp_seed{SEED}.csv',        'mbpp')
print(f'Total corpus: {len(corpus)} | By source: {dict(Counter(r["source"] for r in corpus))}')

extended_humaneval_seed13.csv: 146 rows added
extended_humaneval_x_seed13.csv: 100 rows added
extended_mbpp_seed13.csv: 216 rows added
Total corpus: 462 | By source: {'humaneval': 146, 'humaneval_x': 100, 'mbpp': 216}


In [ ]:
# embedding model (CPU)
import numpy as np
from sentence_transformers import SentenceTransformer

print(f'Loading: {EMBED_MODEL_ID}')
embedder = SentenceTransformer(EMBED_MODEL_ID, device='cpu')
print('Embedding model loaded.')

Loading: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [ ]:
# build / load FAISS indexes (shared across Base RAG and LoRA+RAG runs)
import faiss, pickle

NL_IDX = RAG_DIR / 'nl_index.bin'
PY_IDX = RAG_DIR / 'py_index.bin'
META   = RAG_DIR / 'corpus_meta.pkl'

def _embed(texts, batch=64):
    return embedder.encode(texts, batch_size=batch, show_progress_bar=True,
                           normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

def _build(vecs):
    idx = faiss.IndexFlatIP(vecs.shape[1])
    idx.add(vecs)
    return idx

if NL_IDX.exists() and PY_IDX.exists() and META.exists():
    print('Loading existing FAISS indexes from Drive...')
    nl_index = faiss.read_index(str(NL_IDX))
    py_index = faiss.read_index(str(PY_IDX))
    with open(META,'rb') as f: corpus_meta = pickle.load(f)
    print(f'nl_index: {nl_index.ntotal} | py_index: {py_index.ntotal}')
else:
    print(f'Building FAISS indexes from {len(corpus)} rows...')
    nl_vecs = _embed([r.get('nl','')[:400]          for r in corpus])
    py_vecs = _embed([r.get('py_canonical','')[:500] for r in corpus])
    nl_index = _build(nl_vecs)
    py_index = _build(py_vecs)
    corpus_meta = corpus
    faiss.write_index(nl_index, str(NL_IDX))
    faiss.write_index(py_index, str(PY_IDX))
    with open(META,'wb') as f: pickle.dump(corpus_meta, f)
    print(f'Saved. nl_index: {nl_index.ntotal} vectors, dim={nl_index.d}')

Building FAISS indexes from 462 rows...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Saved. nl_index: 462 vectors, dim=384


In [ ]:
# retrieval function (with self-retrieval guard)
def retrieve(query: str, index, meta: list, k: int, exclude_id: str = None) -> list:
    if k == 0 or not query.strip(): return []
    qvec = embedder.encode([query[:500]], normalize_embeddings=True,
                            convert_to_numpy=True).astype(np.float32)
    fetch_k = min(k + 8, index.ntotal)
    _, positions = index.search(qvec, fetch_k)
    results = []
    for pos in positions[0]:
        if pos < 0 or pos >= len(meta): continue
        row = meta[pos]
        if exclude_id and str(row.get('id','')) == str(exclude_id): continue
        results.append(row)
        if len(results) == k: break
    return results

_s = retrieve('find minimum element in list', nl_index, corpus_meta, k=2)
print(f'Retrieval smoke test (k=2): {len(_s)} results')
for r in _s: print(f'  [{r["source"]}] {r["id"]}: {r.get("nl","")[:55]}')

Retrieval smoke test (k=2): 2 results
  [mbpp] 410: Write a function to find the minimum value in a given h
  [mbpp] 62: Write a python function to find smallest number in a li


## 6. RAG Prompt Builders

In [ ]:
# example formatters
def _fmt_java_ex(retrieved: list) -> str:
    if not retrieved: return ''
    blocks = []
    for i, r in enumerate(retrieved, 1):
        nl   = (r.get('nl') or '').strip()[:300]
        py   = (r.get('py_canonical') or '').strip()[:400]
        java = (r.get('java_canonical') or '').strip()[:500]
        if not java: continue
        blocks.append(
            f'# --- Example {i} ---\n'
            f'# Description: {nl}\n'
            f'```python\n{py}\n```\n'
            f'```java\n{java}\n```'
        )
    return '\n\n'.join(blocks)

def _fmt_nl_ex(retrieved: list) -> str:
    if not retrieved: return ''
    blocks = []
    for i, r in enumerate(retrieved, 1):
        py   = (r.get('py_canonical') or '').strip()[:400]
        java = (r.get('java_canonical') or '').strip()[:400]
        nl   = (r.get('nl') or '').strip()[:300]
        if not nl: continue
        blocks.append(
            f'# --- Example {i} ---\n'
            f'```python\n{py}\n```\n'
            f'```java\n{java}\n```\n'
            f'# Description: {nl}'
        )
    return '\n\n'.join(blocks)

In [ ]:
# task prompt builders
def _prefix(ex, label):
    return f'# {label}:\n\n{ex}\n\n# --- Your Task ---\n' if ex else ''

def p_nl2py(nl, sig, retrieved):
    ex = _fmt_java_ex(retrieved)
    return _prefix(ex, 'Few-shot examples') + (
        f'Implement the following specification in Python.\n'
        f'Return ONLY the function inside a single ```python``` block.\n\n'
        f'# Specification\n{nl}\n\n# Required signature\n```python\n{sig}\n```'
    )

def p_nl2java(nl, decl, retrieved):
    ex = _fmt_java_ex(retrieved)
    d  = f'\n\n# Required Java declaration\n```java\n{decl}\n```' if decl.strip() else ''
    return _prefix(ex, 'Few-shot examples') + (
        f'Implement in Java. Return ONE ```java``` block: public class Solution with public static method.\n\n'
        f'# Specification\n{nl}{d}'
    )

def p_py2java(nl, py, decl, retrieved):
    ex = _fmt_java_ex(retrieved)
    d  = f'\n\n# Required Java signature\n```java\n{decl}\n```' if decl.strip() else ''
    return _prefix(ex, 'Similar Python→Java examples') + (
        f'Translate Python to Java. Return ONE ```java``` block: public class Solution, public static method.\n\n'
        f'# Description\n{nl}\n\n# Python\n```python\n{py}\n```{d}'
    )

def p_py2java_with_test(nl, py, py_tests, retrieved):
    ex = _fmt_java_ex(retrieved)
    return _prefix(ex, 'Similar examples') + (
        f'Translate Python to Java AND write a Main.java test driver.\n'
        f'Return EXACTLY TWO ```java``` blocks: Solution.java then Main.java.\n'
        f'Main uses plain Java assert (no JUnit). Float: Math.abs(a-b)<1e-6.\n\n'
        f'# Description\n{nl}\n\n'
        f'# Python\n```python\n{py}\n```\n\n'
        f'# Python tests (translate to Java assert)\n```python\n{py_tests}\n```'
    )

def p_code2nl(py, java, retrieved):
    ex = _fmt_nl_ex(retrieved)
    return _prefix(ex, 'Similar code+description examples') + (
        f'Write a SINGLE-PARAGRAPH specification (<=120 words) for this function.\n'
        f'Mention inputs, output, edge cases. No code, no markdown.\n\n'
        f'# Python\n```python\n{py}\n```\n\n'
        f'# Java\n```java\n{java}\n```\n\nReturn only the paragraph.'
    )

def p_feedback(original, prev, diff, err):
    cap = lambda s, n=3000: s[:n]+'\n...' if len(s)>n else s
    return (
        f'Previous attempt failed. Correct it in the SAME format.\n\n'
        f'# What was wrong\n{cap(diff,1200)}\n\n'
        f'# Your previous attempt\n{cap(prev)}\n\n'
        f'# Validator output\n{cap(err,1200)}\n\n'
        f'# Original task\n{cap(original)}'
    )

print('Prompt builders defined.')

Prompt builders defined.


## 7. Load Model

Loads Qwen 1.5B base model, then optionally loads LoRA adapter on top.

- `USE_LORA=False` → base model → produces **Base RAG** results
- `USE_LORA=True` → base + adapter → produces **LoRA+RAG** results

Change `USE_LORA` in cell 1.3 between runs.

In [ ]:
# load model (base + optional LoRA adapter)
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
)

# Step 1 — load base model
print(f'Loading base model: {MODEL_ID}')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True,
)

# Step 2 — optionally load LoRA adapter on top
if USE_LORA:
    if ADAPTER_DIR.exists() and (ADAPTER_DIR / 'adapter_config.json').exists():
        print(f'Loading LoRA adapter from {ADAPTER_DIR}...')
        model = PeftModel.from_pretrained(model, str(ADAPTER_DIR))
        print('LoRA adapter loaded — running LoRA+RAG')
    else:
        print(f' Adapter not found at {ADAPTER_DIR}')
        print('   Copy lora_finetuned/ folder to Drive codegen_week1/ first.')
        raise FileNotFoundError(f'LoRA adapter missing: {ADAPTER_DIR}')
else:
    print('USE_LORA=False — running base model RAG')

model.eval()
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f'\nModel : {MODEL_ID}')
print(f'Mode  : {"LoRA+RAG" if USE_LORA else "Base RAG"}  (RUN_TAG={RUN_TAG})')
print(f'VRAM  : {free/1e9:.1f} / {total/1e9:.1f} GB')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading LoRA adapter from /content/drive/MyDrive/codegen_week1/lora_finetuned...
✅ LoRA adapter loaded — running LoRA+RAG

Model : Qwen/Qwen2.5-Coder-1.5B-Instruct
Mode  : LoRA+RAG  (RUN_TAG=lora_rag)
VRAM  : 14.3 / 15.6 GB


## 8. Flow Functions — One Per Task

Each function retrieves K examples, builds RAG prompt, generates with retry loop.
K=0 → zero-shot (no retrieval). K=1 → RAG.

In [ ]:
# Flow 1: NL → Python
def run_nl2py(row: dict, k: int) -> dict:
    nl  = row['nl']
    sig = py_signature(row['py_canonical'])
    retr = retrieve(nl, nl_index, corpus_meta, k=k, exclude_id=row['id'])
    orig = p_nl2py(nl, sig, retr)

    def gen(fb):
        prompt = p_feedback(orig, **fb) if fb else orig
        return orig, {'code': extract_python_body(llm(tokenizer, model, prompt, 512))}

    def val(s):
        g = s.get('code','')
        if not g: return {'passed': False, 'error': 'no code'}
        if row.get('py_entry') and row.get('py_test'):
            full = g + '\n' + row['py_test'] + f"\ncheck({row['py_entry']})\n"
        elif row.get('py_test'):
            full = g + '\n' + row['py_test']
        else: return {'passed': False, 'error': 'no test'}
        return run_python(full)

    def fb(s, err):
        return {'prev': f'```python\n{s.get("code","")}\n```',
                'diff': 'Python test failed. Fix logic or signature.', 'err': err}

    res = run_with_retries(gen, val, fb)
    b   = res['best']
    return {'dataset': row['source'], 'problem_id': row['id'], 'task': 'nl2py', 'k': k,
            'flag': res['flag'], 'n_attempts': res['n_attempts'],
            'prediction': b['struct'].get('code','')[:800],
            'fail_reason': '' if res['flag'] else (b['verdict'].get('error','') or '')[:300]}

In [ ]:
#  Flow 2: NL → Java
def run_nl2java(row: dict, k: int) -> dict:
    nl        = row['nl']
    java_decl = row.get('java_decl', '')
    java_test = row.get('java_test', '')
    retr = retrieve(nl, nl_index, corpus_meta, k=k, exclude_id=row['id'])
    orig = p_nl2java(nl, java_decl, retr)

    def gen(fb):
        prompt = p_feedback(orig, **fb) if fb else orig
        return orig, {'code': extract_java_body(llm(tokenizer, model, prompt, 900))}

    def val(s):
        g = s.get('code','')
        if not g: return {'passed': False, 'error': 'no code'}
        if not java_test: return {'passed': False, 'error': 'no java_test'}
        return validate_java(g, java_test)

    def fb(s, err):
        return {'prev': f'```java\n{s.get("code","")}\n```',
                'diff': 'javac or java -ea failed. Fix types, static modifier.', 'err': err}

    res = run_with_retries(gen, val, fb)
    b   = res['best']
    return {'dataset': row['source'], 'problem_id': row['id'], 'task': 'nl2java', 'k': k,
            'flag': res['flag'], 'n_attempts': res['n_attempts'],
            'prediction': b['struct'].get('code','')[:800],
            'fail_reason': '' if res['flag'] else (b['verdict'].get('error','') or '')[:300]}

In [ ]:
# Flow 3: Python → Java
def run_py2java(row: dict, k: int) -> dict:
    nl        = row['nl']
    py        = row['py_canonical']
    java_decl = row.get('java_decl', '')
    java_test = row.get('java_test', '')
    retr = retrieve(py, py_index, corpus_meta, k=k, exclude_id=row['id'])

    if java_test.strip():
        orig = p_py2java(nl, py, java_decl, retr)
        def gen(fb):
            prompt = p_feedback(orig, **fb) if fb else orig
            return orig, {'java_solution': extract_java_body(llm(tokenizer, model, prompt, 900))}
        def val(s):
            g = s.get('java_solution','')
            if not g: return {'passed': False, 'error': 'no code'}
            return validate_java(g, java_test)
        def fb(s, err):
            return {'prev': f'```java\n{s.get("java_solution","")}\n```',
                    'diff': 'javac or java -ea failed. Fix types, static modifier.', 'err': err}
        key = 'java_solution'
    else:
        py_tests = row.get('py_test', '')
        orig = p_py2java_with_test(nl, py, py_tests, retr)
        def gen(fb):
            prompt = p_feedback(orig, **fb) if fb else orig
            raw = llm(tokenizer, model, prompt, 1400)
            sol, tst = extract_solution_and_test(raw)
            return orig, {'java_solution': sol, 'java_test_gen': tst}
        def val(s):
            sol = s.get('java_solution','')
            tst = s.get('java_test_gen','')
            if not sol: return {'passed': False, 'error': 'no solution'}
            if not tst: return {'passed': False, 'error': 'no test generated'}
            return validate_java(sol, tst)
        def fb(s, err):
            prev = (f'```java\n// Solution\n{s.get("java_solution","")}\n```\n'
                    f'```java\n// Main\n{s.get("java_test_gen","")}\n```')
            return {'prev': prev, 'diff': 'Fix assert syntax, static modifier.', 'err': err}
        key = 'java_solution'

    res = run_with_retries(gen, val, fb)
    b   = res['best']
    return {'dataset': row['source'], 'problem_id': row['id'], 'task': 'py2java', 'k': k,
            'flag': res['flag'], 'n_attempts': res['n_attempts'],
            'prediction': b['struct'].get(key,'')[:800],
            'fail_reason': '' if res['flag'] else (b['verdict'].get('error','') or '')[:300]}

In [ ]:
# Flow 4: Code → NL (ROUGE-L)
from rouge_score import rouge_scorer as _rs
_rouge = _rs.RougeScorer(['rougeL'], use_stemmer=True)

def rougeL(ref, hyp):
    if not ref or not hyp: return 0.0
    return _rouge.score(ref, hyp)['rougeL'].fmeasure

def run_code2nl(row: dict, k: int) -> dict:
    py   = strip_py_docstring(row['py_canonical'])
    java = row.get('java_canonical', '')
    ref  = row['nl'].strip()
    retr = retrieve(py, py_index, corpus_meta, k=k, exclude_id=row['id'])
    prompt = p_code2nl(py, java, retr)
    raw    = llm(tokenizer, model, prompt, 400)
    pred   = strip_markdown(raw)
    score  = rougeL(ref, pred)
    return {'dataset': row['source'], 'problem_id': row['id'], 'task': 'code2nl', 'k': k,
            'flag': None, 'n_attempts': 1,
            'rougeL': round(score, 4),
            'prediction': pred[:500], 'fail_reason': ''}

TASK_FNS = {'nl2py': run_nl2py, 'nl2java': run_nl2java,
            'py2java': run_py2java, 'code2nl': run_code2nl}
print('Flow functions + TASK_FNS defined.')

Flow functions + TASK_FNS defined.


## 9. Evaluation — All Tasks × All K Values

Saves after **each task** — safe to resume on disconnect.

K=0 = zero-shot · K=1 = RAG · Results tagged with `RUN_TAG` (base_rag or lora_rag).

In [ ]:
# evaluation loop with per-task save and resume
import pandas as pd
from tqdm.auto import tqdm

all_rows   = []
all_scores = {}

DATASETS = [('humaneval', he_eval), ('mbpp', mbpp_eval)]

for k in K_VALUES:
    tag = f'{RUN_TAG}_k{k}'   # e.g. base_rag_k0, lora_rag_k1
    print(f'\n{"="*60}')
    print(f'  K={k}  ({"zero-shot" if k==0 else f"RAG K={k}"})  [{RUN_TAG}]')
    print(f'{"="*60}')
    all_scores[k] = {}

    for ds_name, ds_eval in DATASETS:
        if not ds_eval:
            print(f'Skipping {ds_name}')
            continue
        all_scores[k][ds_name] = {}
        print(f'\n  {ds_name} ({len(ds_eval)} problems × 4 tasks)')

        for task_name, task_fn in TASK_FNS.items():
            # Resume support — check if already done
            save_path = RAG_DIR / f'rag_{tag}_{ds_name}_{task_name}.csv'
            if save_path.exists():
                df_done   = pd.read_csv(save_path)
                done_pids = set(df_done['problem_id'].astype(str))
                if len(done_pids) >= len(ds_eval):
                    score = df_done['rougeL'].mean() if task_name=='code2nl' else df_done['flag'].mean()
                    all_scores[k][ds_name][task_name] = round(score, 4)
                    all_rows.extend(df_done.to_dict('records'))
                    print(f'    {task_name}: LOADED {score:.4f} from Drive')
                    continue
            else:
                done_pids = set()

            # Run remaining problems
            remaining  = [p for p in ds_eval if str(p['id']) not in done_pids]
            task_rows  = []
            for row in tqdm(remaining, desc=f'  {tag}:{ds_name}:{task_name}'):
                result = task_fn(row, k)
                task_rows.append(result)
                all_rows.append(result)

            # Save immediately after task completes
            df_task = pd.DataFrame(task_rows)
            if save_path.exists():
                df_task.to_csv(save_path, mode='a', header=False, index=False)
            else:
                df_task.to_csv(save_path, index=False)

            score  = df_task['rougeL'].mean() if task_name=='code2nl' else df_task['flag'].mean()
            all_scores[k][ds_name][task_name] = round(score, 4)
            metric = 'ROUGE-L' if task_name=='code2nl' else 'pass@1'
            print(f'    {task_name}: {score:.4f} ({metric}) → saved')

print('\nAll evaluations complete.')


  K=0  (zero-shot)  [lora_rag]

  humaneval (33 problems × 4 tasks)


  lora_rag_k0:humaneval:nl2py:   0%|          | 0/33 [00:00<?, ?it/s]

    nl2py: 0.5455 (pass@1) → saved


  lora_rag_k0:humaneval:nl2java:   0%|          | 0/33 [00:00<?, ?it/s]

    nl2java: 0.6667 (pass@1) → saved


...

    nl2java: 0.2424 (pass@1) → saved


  lora_rag_k1:mbpp:py2java:   0%|          | 0/33 [00:00<?, ?it/s]

    py2java: 0.7576 (pass@1) → saved


  lora_rag_k1:mbpp:code2nl:   0%|          | 0/33 [00:00<?, ?it/s]

    code2nl: 0.1852 (ROUGE-L) → saved

All evaluations complete.


## 10. Results

In [ ]:
# per-dataset results table
import pandas as pd

TASK_META = {
    'nl2py':   ('NL→Python',   'pass@1'),
    'nl2java': ('NL→Java',     'pass@1'),
    'py2java': ('Python→Java', 'pass@1'),
    'code2nl': ('Code→NL',     'ROUGE-L'),
}

rows = []
for ds_name, ds_eval in DATASETS:
    if not ds_eval: continue
    for task, (label, metric) in TASK_META.items():
        zs  = all_scores.get(0,{}).get(ds_name,{}).get(task, float('nan'))
        k1  = all_scores.get(1,{}).get(ds_name,{}).get(task, float('nan'))
        d   = round(k1-zs, 4) if (k1==k1 and zs==zs) else float('nan')
        rows.append({'run': RUN_TAG, 'dataset': ds_name, 'task': label,
                     'metric': metric, 'n_problems': len(ds_eval),
                     'zero_shot': zs, 'rag_k1': k1, 'delta': d})

df_res = pd.DataFrame(rows)
df_res.to_csv(RAG_DIR / f'rag_per_dataset_{RUN_TAG}.csv', index=False)
print(f'=== Per Dataset ({RUN_TAG}) ===')
print(df_res.to_string(index=False))

=== Per Dataset (lora_rag) ===
     run   dataset        task  metric  n_problems  zero_shot  rag_k1   delta
lora_rag humaneval   NL→Python  pass@1          33     0.5455  0.5455  0.0000
lora_rag humaneval     NL→Java  pass@1          33     0.6667  0.6364 -0.0303
lora_rag humaneval Python→Java  pass@1          33     0.7273  0.6667 -0.0606
lora_rag humaneval     Code→NL ROUGE-L          33     0.7731  0.7690 -0.0041
lora_rag      mbpp   NL→Python  pass@1          33     0.6667  0.7273  0.0606
lora_rag      mbpp     NL→Java  pass@1          33     0.1212  0.2424  0.1212
lora_rag      mbpp Python→Java  pass@1          33     0.8485  0.7576 -0.0909
lora_rag      mbpp     Code→NL ROUGE-L          33     0.1970  0.1852 -0.0118


In [ ]:
# combined summary
combined = {}
for task, (label, metric) in TASK_META.items():
    tot_zs, tot_k1, tot_n = 0.0, 0.0, 0
    for ds_name, ds_eval in DATASETS:
        if not ds_eval: continue
        n  = len(ds_eval)
        zs = all_scores.get(0,{}).get(ds_name,{}).get(task, 0.0)
        k1 = all_scores.get(1,{}).get(ds_name,{}).get(task, 0.0)
        if zs==zs: tot_zs += zs*n
        if k1==k1: tot_k1 += k1*n
        tot_n += n
    combined[task] = {'run': RUN_TAG, 'task': label, 'metric': metric,
                      'n_total': tot_n,
                      'zero_shot': round(tot_zs/max(1,tot_n),4),
                      'rag_k1':    round(tot_k1/max(1,tot_n),4),
                      'delta':     round((tot_k1-tot_zs)/max(1,tot_n),4)}

df_comb = pd.DataFrame(combined.values())
df_comb.to_csv(RAG_DIR / f'rag_combined_summary_{RUN_TAG}.csv', index=False)
print(f'=== Combined Summary ({RUN_TAG}) ===')
print(df_comb.to_string(index=False))

=== Combined Summary (lora_rag) ===
     run        task  metric  n_total  zero_shot  rag_k1   delta
lora_rag   NL→Python  pass@1       66     0.6061  0.6364  0.0303
lora_rag     NL→Java  pass@1       66     0.3939  0.4394  0.0455
lora_rag Python→Java  pass@1       66     0.7879  0.7121 -0.0758
lora_rag     Code→NL ROUGE-L       66     0.4850  0.4771 -0.0080


In [1]:
# 4-way comparison: Zero-shot | Base RAG | LoRA | LoRA+RAG
import json

print('=== 4-WAY COMPARISON ===')
print()

# Load LoRA-only scores
lora_scores = {}
for fn in ['lora_comparison_aditi_mbppaug.csv', 'lora_comparison_aditi.csv', 'lora_results.csv']:
    lp = RESULTS_DIR / fn
    if lp.exists():
        ldf = pd.read_csv(lp)
        tmap = {'NL→Python':'nl2py','NL→Java':'nl2java',
                'Python→Java':'py2java','Code→NL':'code2nl'}
        for _, r in ldf.iterrows():
            k = tmap.get(r.get('task',''))
            if k:
                lora_scores[k] = r.get(f'qlora_{MODEL_SIZE}',
                               r.get('qlora_1.5b',
                               r.get('qlora_7b', float('nan'))))
        print(f'LoRA-only scores loaded from {fn}')
        break
else:
    print('LoRA CSV not found — LoRA column will show nan')

# Load base RAG scores if running LoRA+RAG
base_rag_scores = {}
base_rag_csv = RAG_DIR / 'rag_combined_summary_base_rag.csv'
if base_rag_csv.exists():
    bdf = pd.read_csv(base_rag_csv)
    tmap2 = {'NL→Python':'nl2py','NL→Java':'nl2java',
             'Python→Java':'py2java','Code→NL':'code2nl'}
    for _, r in bdf.iterrows():
        k = tmap2.get(r.get('task',''))
        if k: base_rag_scores[k] = {'zero_shot': r.get('zero_shot'), 'rag_k1': r.get('rag_k1')}
    print(f'Base RAG scores loaded from rag_combined_summary_base_rag.csv')
else:
    print('Base RAG CSV not found — run with USE_LORA=False first')

print()
h = f'{"Task":<16} {"Metric":<12} {"Zero-shot":>10} {"Base RAG":>10} {"LoRA":>10} {"LoRA+RAG":>10}'
print(h)
print('-'*len(h))

for task, (label, metric) in TASK_META.items():
    zs       = base_rag_scores.get(task, {}).get('zero_shot', combined.get(task,{}).get('zero_shot', float('nan')))
    base_rag = base_rag_scores.get(task, {}).get('rag_k1', float('nan'))
    lora     = lora_scores.get(task, float('nan'))
    lora_rag = combined.get(task, {}).get('rag_k1', float('nan'))
    print(f'{label:<16} {metric:<12} {zs:>10.4f} {base_rag:>10.4f} {lora:>10.4f} {lora_rag:>10.4f}')

print()
print('Expected: Zero-shot < Base RAG ≈ LoRA < LoRA+RAG')

=== 4-WAY COMPARISON ===



NameError: name 'RESULTS_DIR' is not defined

## 11. Unload Model & Save

In [ ]:
import gc, torch, shutil

for f in RAG_DIR.glob(f'rag_*{RUN_TAG}*.csv'):
    shutil.copy(f, RESULTS_DIR / f.name)
    print(f'Saved {f.name}')

del model, tokenizer
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f'GPU freed: {free/1e9:.1f} / {total/1e9:.1f} GB')

Saved rag_lora_rag_k0_humaneval_nl2py.csv
Saved rag_lora_rag_k0_humaneval_nl2java.csv
Saved rag_lora_rag_k0_humaneval_py2java.csv
Saved rag_lora_rag_k0_humaneval_code2nl.csv
Saved rag_lora_rag_k0_mbpp_nl2py.csv
Saved rag_lora_rag_k0_mbpp_nl2java.csv
Saved rag_lora_rag_k0_mbpp_py2java.csv
Saved rag_lora_rag_k0_mbpp_code2nl.csv
Saved rag_lora_rag_k1_humaneval_nl2py.csv
Saved rag_lora_rag_k1_humaneval_nl2java.csv
Saved rag_lora_rag_k1_humaneval_py2java.csv
Saved rag_lora_rag_k1_humaneval_code2nl.csv
Saved rag_lora_rag_k1_mbpp_nl2py.csv
Saved rag_lora_rag_k1_mbpp_nl2java.csv
Saved rag_lora_rag_k1_mbpp_py2java.csv
Saved rag_lora_rag_k1_mbpp_code2nl.csv
Saved rag_per_dataset_lora_rag.csv
Saved rag_combined_summary_lora_rag.csv
GPU freed: 14.4 / 15.6 GB


## Done

**Run order:**
1. Set `USE_LORA = False` in cell 1.3 → run all cells → produces `base_rag` results
2. Set `USE_LORA = True` in cell 1.3 → run all cells → produces `lora_rag` results
3. Cell 10.3 automatically shows the 4-way comparison table

**Output files (rag_final/ and results/):**
- `rag_combined_summary_base_rag.csv` — Base model + RAG
- `rag_combined_summary_lora_rag.csv` — LoRA fine-tuned + RAG
- `rag_per_dataset_base_rag.csv` / `rag_per_dataset_lora_rag.csv` — per dataset breakdown
- `rag_{tag}_{dataset}_{task}.csv` — per-task raw results (resume safe)

**Adapter setup:** Copy `lora_finetuned/` folder to Drive `codegen_week1/lora_finetuned/`

**Circular risk note:** MBPP NL→Java and Python→Java use model-generated java_test from extended CSV.